# 표준 Notebook 샘플 - Wine 등급 분류

iris와는 완전히 다른 팀/다른 모델(RandomForest)이지만, 실행 방식은 동일하다.
**이 Notebook을 위해 새로운 Airflow DAG를 만들 필요가 없다** - 같은
`mlops_notebook_executor` DAG에 `code_key`만 바꿔서 넘기면 된다.

In [ ]:
output_dir = "./output"
mlflow_tracking_uri = "http://mlflow:5000"
dataset_key = "demo.wine"
dataset_version = "v1"
model_key = "demo.wine_model"


### ① MLOps SDK Import

In [ ]:
import os
import prizm
import pandas as pd
import joblib
import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


### ② Data Asset 다운로드

In [ ]:
dataset_path = prizm.dataset.download(dataset_key, dataset_version)
df = pd.read_csv(dataset_path)
df.head()


### ③ Hyperparameter

In [ ]:
test_size = 0.2
random_state = 42
n_estimators = 200


### ④ Data Split

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state
)


### ⑤ Training

In [ ]:
clf = RandomForestClassifier(n_estimators=n_estimators, random_state=random_state)
clf.fit(X_train, y_train)


### ⑥ Evaluation

In [ ]:
acc = accuracy_score(y_test, clf.predict(X_test))
print(f"accuracy: {acc:.4f}")


### ⑦ Model Save / Asset 등록

In [ ]:
os.makedirs(output_dir, exist_ok=True)
model_path = os.path.join(output_dir, "model.joblib")
joblib.dump(clf, model_path)

mlflow.set_tracking_uri(mlflow_tracking_uri)
mlflow.set_experiment("mlops-core-demo")

with mlflow.start_run(run_name="wine-training"):
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_artifact(model_path)

    result = prizm.model.upload(model_key, model_path)
    mlflow.set_tag("mlops_asset_key", result["key"])
    mlflow.set_tag("mlops_asset_version", result["version"])

print("registered:", result)
